**Latest hydraulic diagnostic: common caps 0.05–5 m/s for all 16 base networks.** See the final section for the cap comparison, three clipped-link exceptions and revised distributions. Earlier cap scenarios remain historical comparisons. Production settings are unchanged.

**Latest decision: ROBUST-SLOPES-2026-10-02.** The final section contains the selected robust method, gap constraints, all-16 results and hydraulic limitations. Earlier sections are retained as the investigation history. See also [the full methodology](../ROBUST_REFERENCE_PROFILE_METHOD.md).

# Smoothing, minimum slope and unit allocation: controlled comparison

**2 October 2026 — completed across all 16 base networks.** The previous section-floor candidate changed too many things together. This experiment removes section targets, re-selects smoothing with the converged solver, and compares profile floors 0.01 and 0.05 × the static reference slope. Both profiles receive the old and local-target graph allocations. A third allocation control keeps the old domain-wide target while changing only the lower-envelope rule.

**32 selected profiles, 96 completed base allocations, 331 lambda candidates (three whole-cycle folds each).** No production slopes, resistance values, Kb settings or hydrographs changed. The case-46 manual bridge is retained in all fold/final profiles.

[Full methods and findings](../assets/smoothing_floor_allocation_20261002/README.md). The suggested next candidate is re-selected smoothing, 0.05 profile floor, and the old unit allocation. This recommendation concerns the tested fit/low-slope trade-off; it does not establish a physical minimum or routing stability.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src').is_dir())
OUT=ROOT/'RAPID/assets/smoothing_floor_allocation_20261002'
meta=json.loads((OUT/'method.json').read_text());sources=meta['sources']
def read(name):
    t=pd.read_csv(OUT/name,dtype={'example':str})
    if 'example' in t:t.example=t.example.str.zfill(2)
    return t
profiles=read('profiles.csv');allocations=read('allocations.csv');cv=read('all_cv_candidates.csv')
assert len(profiles)==32 and len(allocations)==96 and allocations.status.eq('complete').all()
pio.renderers.default='plotly_mimetype'
labels={'old':'Domain target + old 1% unit floor','maxmin':'Local-gradient target + graph minimum','maxmin_domain':'Domain target + graph minimum'}
display(pd.read_csv(OUT/'comparison.csv').replace({'allocation':labels}))
display(profiles[['example','floor','selected_lambda','lambda_ratio','cv_rmse','head_change_pct','selected_grid_edge']])

## How to interpret the comparison

**Smoothing:** lambda is selected separately for each floor using identical three-fold whole-cycle splits. Test acquisitions receive fitted constant offsets, so scores assess longitudinal shape, not prediction of absolute water level. Each cycle contributes equally to scoring and to training weight. These are model-selection scores, not a separate final test set. Numerically tied minima (within 1e-6 relative MSE) favor the larger lambda; case 15 has a practically flat low-lambda optimum.

**Floor:** 0.05 is a sensitivity choice, not a universal physical threshold. The effect on held-out RMSE ranges from approximately −0.126% to +0.422%, with the largest increase in case 11. This small difference does not prove that the higher floor is physically preferable.

**Unit allocation:** `old` uses the existing domain-wide reference slope and 1% domain floor. `maxmin` uses local fitted gradients as targets and a graph-derived common minimum fraction. `maxmin_domain` uses the old domain-wide target with that graph-derived minimum; it isolates target variation from the envelope rule. Every method preserves the SAME boundary heads for a given profile.

The local-target method produces substantially more low-slope links. A graph minimum only protects slopes relative to the supplied targets; flat local targets can still create flat branches. The constant-domain control offers limited overall benefit and has not been independently validated against branch observations.

In [ ]:
display(allocations.pivot(index='example',columns=['floor','allocation'],values='below_1e5'))
display(read('minimum_links.csv').query("floor == .05 and allocation == 'old'").sort_values('slope'))
errors=profiles.pivot(index='example',columns='floor',values='cv_rmse')
display((100*(errors[.05]/errors[.01]-1)).rename('higher_floor_RMSE_change_pct').to_frame())

In [ ]:
def inspect(ex):
    old=pd.read_parquet(Path(sources[ex])/'reference_profile.parquet')
    obs=pd.read_parquet(Path(sources[ex])/'reference_profile_observations.parquet')
    fig=go.Figure()
    for aid,g in obs.groupby('acquisition_id',sort=False):
        g=g.sort_values('reference_section_dist_out_m');r=g.iloc[0]
        fig.add_scatter(x=g.reference_section_dist_out_m/1000,y=g.wse_m,mode='lines',name=f"Cycle {int(r.cycle):03d} · pass {int(r['pass']):03d}",visible='legendonly',opacity=.4,line=dict(width=1))
    fig.add_scatter(x=old.reference_section_dist_out_m/1000,y=old.fitted_wse_m,name='Previous production profile',line=dict(color='#555555',dash='dash'))
    for eta,color in [(.01,'#2563eb'),(.05,'#f97316')]:
        f=pd.read_parquet(OUT/ex/f'floor_{eta:.2f}'/'profile.parquet')
        fig.add_scatter(x=f.reference_section_dist_out_m/1000,y=f.fitted_wse_m,name=f'Re-selected smoothing; floor {eta}',line=dict(color=color,width=2.5))
    fig.update_layout(title=f'Case {ex}: reference profiles',xaxis_title='Distance to outlet (km)',yaxis_title='WSE (m)',height=500,template='plotly_white');fig.show()
    fig=go.Figure()
    for eta,color in [(.01,'#2563eb'),(.05,'#f97316')]:
        g=cv[(cv.example==ex)&(cv.floor==eta)&(cv.status=='complete')].sort_values('lambda_value')
        fig.add_scatter(x=g.lambda_value,y=np.sqrt(g.cv_mse),mode='lines+markers',name=f'Floor {eta}',line=dict(color=color))
        selected=profiles[(profiles.example==ex)&(profiles.floor==eta)].iloc[0]
        fig.add_scatter(x=[selected.selected_lambda],y=[selected.cv_rmse],mode='markers',marker=dict(symbol='star',size=12,color=color),showlegend=False)
    fig.update_layout(title='Whole-cycle cross-validation (stars = selected lambda)',xaxis_title='Curvature strength lambda',xaxis_type='log',yaxis_title='Held-out cycle RMSE (m)',height=350,template='plotly_white');fig.show()
    fig=go.Figure()
    for eta,method,color in [(.01,'old','#777777'),(.05,'old','#f97316'),(.05,'maxmin_domain','#16a34a'),(.05,'maxmin','#2563eb')]:
        d=OUT/ex/f'floor_{eta:.2f}'/method;l=pd.read_parquet(d/'links.parquet');n=pd.read_parquet(d/'nodes.parquet').set_index('id_node')
        mid=(l.id_us_node.map(n.reference_section_dist_out_m)+l.id_ds_node.map(n.reference_section_dist_out_m))/2000
        fig.add_scatter(x=mid,y=l.slope,mode='markers',name=f'{eta}: {labels[method]}',visible=True if method=='old' else 'legendonly',marker=dict(size=5,color=color,opacity=.65),customdata=np.c_[l.id_link,l.ownership_kind],hovertemplate='Link %{customdata[0]} · %{customdata[1]}<br>Slope %{y:.5g}<extra></extra>')
    fig.add_hline(y=1e-5,line_dash='dot',line_color='gray')
    fig.update_layout(title='Physical link slopes after unit allocation',xaxis_title='Projected midpoint (km)',yaxis_title='Slope (m/m)',yaxis_type='log',height=400,template='plotly_white');fig.show()
    display(profiles[profiles.example==ex][['floor','cv_rmse','head_change_pct','in_sample_cycle_rmse','selected_lambda','min_profile_slope']])
    display(allocations[allocations.example==ex][['floor','allocation','below_1e5','below_1e6','min_slope','max_floor_violation_m']])

## Case 02

In [ ]:
inspect("02")

## Case 03

In [ ]:
inspect("03")

## Case 05

In [ ]:
inspect("05")

## Case 11

In [ ]:
inspect("11")

## Case 13

In [ ]:
inspect("13")

## Case 15

In [ ]:
inspect("15")

## Case 21

In [ ]:
inspect("21")

## Case 22

In [ ]:
inspect("22")

## Case 37

In [ ]:
inspect("37")

## Case 41

In [ ]:
inspect("41")

## Case 45

In [ ]:
inspect("45")

## Case 46

In [ ]:
inspect("46")

## Case 48

In [ ]:
inspect("48")

## Case 54

In [ ]:
inspect("54")

## Case 55

In [ ]:
inspect("55")

## Case 56

In [ ]:
inspect("56")

## Where this leaves us

The simpler re-selected profile with the old allocation is the preferred next candidate for review. The 0.05 floor removes all sub-1e-6 links in this test; its minimum is 3.60e-6. It still has 47 links below 1e-5, including 28 in case 46. The profile floor and physical branch slopes are different quantities, so a profile floor is not an absolute branch-slope threshold.

Review case 21's ~13.5% head-loss change and the remaining weak/flat sections before using these candidates for resistance or Kb/routing checks. The graph-domain alternative can stay a separate sensitivity. The local-gradient-target variant should not be adopted based on this experiment. Production remains unchanged.

# Review of the selected slope candidate — 2026-10-02

## Why this review was made
The smoothing/floor experiment proposed re-selected smoothing, equal-cycle weighting, a reference-profile floor of 0.05 S_ref and the previous unit allocation. Its minimum final link slope is 3.603045e-6 m/m, with 47 links below 1e-5 and none below 1e-6. The review asks whether the candidate respects observed local gradients, not merely whether its minimum looks acceptable. No production files or routing runs were changed.

## Conclusion
Retain this as an experimental candidate, but **do not promote all 16 profiles yet**. Cases 13, 21, 22 and 46 have identifiable local concerns. Cases 11, 15 and 45 need explicit uncertainty/allocation qualifications. Other cases can proceed to hydraulic diagnostics, with the local discrepancies below retained in the audit. This is a review classification, not independent validation or a claim of physical truth.

## All 16 cases
| Case | Links below 1e-5 | Review finding / disposition |
|---|---:|---|
| 02 | 0 | Proceed to diagnostics. Candidate falls within cycle P10–P90 in all 12 eligible 10 km windows; previous profile missed 3. |
| 03 | 0 | Proceed. All 3 eligible windows within cycle range; total drop changes only +0.15 m. |
| 05 | 0 | Proceed. All 18 eligible windows within range; total drop essentially unchanged. |
| 11 | 3 | Conditional. About 81% of profile length is floor-active. Remaining low links have mixed-sign or insufficient local cycle support. Candidate total drop 14.96 m is within endpoint-cycle diagnostic range 12.80–16.55 m, but two windows disagree locally. Do not describe its flat sections as directly measured positive slopes. |
| 13 | 1 | Hold local correction: link 93 at 63.56–66.17 km is newly too flat relative to all 7 eligible cycles. Overall window agreement improves strongly (19 previous versus 5 candidate outside range). |
| 15 | 4 | Conditional unit-allocation review. The four lows remain in compound allocation; no eligible exact-span cycle slopes. Whole-corridor agreement is good (all 10 eligible windows within range). Domain-target max-min allocation removes these lows, but is not yet selected globally. |
| 21 | 0 | Hold. Large total head-loss change is mainly across an unobserved gap; also flattening inconsistent with cycles at 40–50 km. |
| 22 | 5 | Hold local correction. 56% floor-active; 110–120 km is substantially flatter than 16 cycles indicate. Overall agreement improves, but local deterioration remains. |
| 37 | 0 | Proceed with local uncertainty. Two adjoining windows at about 51–71 km move outside a narrow cycle range, in opposite directions; possible redistribution of gradient. |
| 41 | 0 | Proceed with local uncertainty. No floor-active profile length; candidate outside cycle range in 13/45 windows versus 22 previously. Remaining deviations require uncertainty reporting, not a higher minimum. |
| 45 | 6 | Conditional. Short low links have strongly variable, sometimes adverse cycle gradients. They do not provide a clear observed positive lower bound. Three of 21 broad windows outside range. |
| 46 | 28 | Hold targeted correction. Consistently steeper cycle slopes at 73.65–83.14 and 425.71–428.45 km; minimum-slope compound domain has sparse, separated observations. 21/64 broad windows outside range versus 15 previously. |
| 48 | 0 | Proceed with minor local uncertainty. Two of eight windows outside very narrow cycle ranges by small amounts. The 5 km endpoint-line extrapolation gives larger total drops than either profile; this diagnostic is sensitive to endpoint curvature and is not a direct measurement. |
| 54 | 0 | Proceed with outlet qualification. Total drop decreases 1.18 m, mainly outlet elevation +1.04 m. One of 13 windows outside range versus seven previously; no same-acquisition full-endpoint diagnostic available. |
| 55 | 0 | Proceed with local uncertainty. Two of 24 windows outside range, no floor activity, total drop +0.29 m. |
| 56 | 0 | Proceed. All eight eligible windows within range versus three previous misses; total drop −0.20 m. |

## Main findings
### Case 21: the extra head loss is not caused by raising the floor
Total drop changes from 44.709 to 50.749 m (+13.51%). Re-selecting smoothing at the old 0.01 floor already gives 50.792 m: raising the floor to 0.05 slightly *reduces* that drop, by 0.043 m.

There are no observations between 89.21 and 103.38 km, and no acquisition spans across this gap. In the 90.04–100.04 km window alone, the fitted drop rises from 1.698 to 7.366 m (+5.668 m), compared with +6.040 m across the whole network. This percentage is a net accounting comparison; changes elsewhere partly cancel. Separate acquisition offsets mean the cross-gap level difference is not directly fixed by a common pass. Smoothing and other constraints choose the bridge; the added drop cannot be presented as directly observed. Endpoint neighbourhoods themselves have observations, so missing endpoint observations are not the primary issue.

There is also a locally suppressed gradient at 40–50 km: 14 cycle slopes have median 9.3e-5 versus candidate matched slope approximately 1.2e-5.

### Cases 13 and 22: minimum constraints do not guarantee local agreement
Case 13 link 93 has slope 9.24e-6. Its seven cycle slopes have P10/median/P90 approximately 4.0e-5 / 6.0e-5 / 9.3e-5; the candidate evaluated at observation locations is approximately 6e-6. This is a direct-profile link, not a compound allocation effect.

Case 22 at 110–120 km has 16 cycle slopes with median 1.24e-4 and P10 1.15e-4; candidate matched slope is approximately 1.1e-5, with 90% of the window floor-active. The previous fit was approximately 1.44e-4. The candidate is clearly less representative of local cycle slopes there.

### Case 46: distinguish three causes
- At 73.65–83.14 km (link 65), 17 cycles have slope P10/median/P90 2.2e-5 / 2.6e-5 / 7.1e-5, while the candidate is near 6.5e-6.
- At 425.71–428.45 km (links 218/219), 15 cycles give 3.4e-5 / 4.4e-5 / 7.4e-5, while the candidate is again near 6.5e-6.
- The minimum link 316 is in compound domain 74: 3.603045e-6 over 4.144 km, only 0.01493 m head loss. The broader 267.93–275.03 km domain has 60 observations at 15 positions from 10 cycles, but they are spatially separated: exact link spans lack eligible slope support. This is sparse support plus profile/allocation dependence, not evidence for a measured minimum slope.

Other low sections have mixed-sign cycle slopes and are compatible with uncertain near-flat water surfaces. None of the remaining low-link spans falls within the manual 370–410 km bridge. That bridge does not explain these 28 links.

## What to test next
Keep the simple floor candidate, but first test whether a robust observation loss or targeted quality screening removes the local suppression in cases 13, 22 and 46. The plots show large within-acquisition steps in some observations; these can influence squared-error fits despite equal total cycle weights. This is a hypothesis to test, not a demonstrated root cause. Compare local cycle slopes as well as whole-cycle CV; the latter alone missed the local deterioration.

Handle case 21's disconnected observation support explicitly: test a documented cross-gap constraint/bridge and quantify its head-loss sensitivity. No amount of choosing a steeper cycle supplies a missing cross-gap measurement. Avoid selecting the bridge solely to obtain hydraulic stability.

After those checks, rerun base unit allocation and the Kb/Manning stability audit. Do not raise the global floor again merely to pass that audit.

## Methods and limitations
- All 16 cases and all 47 final links below 1e-5 were screened.
- Broad windows are 10 km, requiring at least six positions, 70% spatial coverage and maximum internal gap of 30% per acquisition. Robust Theil–Sen slopes are combined by cycle median. At least five eligible cycles are required for reported window flags.
- Exact low-link windows require four positions and 50% coverage; expanded windows add 1 km on each side. Short or fragmented spans often fail eligibility; this does not mean no observations exist.
- P10–P90 describes between-cycle variation, **not a confidence interval**. Falling outside is a review flag, not an automatic rejection. Nonlinear spatial shape and observational support matter.
- Matched fitted slopes use the same observation positions. Reference-corridor slopes and physical branch slopes are distinguished; unit allocation can produce lower physical slopes because branch lengths are longer.
- Profile figures subtract a constant acquisition offset fitted to the candidate. Zooms align each acquisition within the displayed window and align the previous profile at the left boundary. These illustrate shape only, not independent absolute elevation validation.
- Endpoint cycle drops extrapolate separate robust lines from the outer 5 km and require the same acquisition at both ends. They are unavailable for some cases and can be biased by curvature.
- CV results used for smoothing selection are not independent final validation. No hydraulic simulations were run in this review.

## Artifacts
- `cases.csv`: head losses, endpoint support, minimum slopes and floor-active fractions.
- `windows.csv`: broad-window cycle comparisons.
- `low_links.csv`: all 47 low links and local cycle support.
- `endpoint_cycle_drops.csv`: eligible same-acquisition endpoint diagnostics.
- `all_profiles.png`, `all_local_slopes.png`, `priority_sections.png`: review figures.
- Reproduce with `RAPID/investigations/review_selected_slope_cases.py` and `RAPID/investigations/plot_selected_slope_review.py` from repository root.


### Priority local sections

![Priority local sections](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/4c795a2a70a5_priority_sections.png)

### All 16 reference profiles

![All 16 reference profiles](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/a7715aa1ff12_all_profiles.png)

### Matched local slopes across all 16 cases

![Matched local slopes across all 16 cases](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/a2b6d49829d1_all_local_slopes.png)

# Controlled robust-loss test — 2026-10-02

## Purpose and result
Test whether reducing extreme-observation influence improves local slopes without manually adjusting individual cases. All 16 profiles and base-network unit allocations were recalculated in isolated outputs. Production files remain unchanged.

**Promising controlled result, not yet a validated replacement.** Links below 1e-5 fall from 47 to 31. The minimum remains 3.603045e-6; no links fall below 1e-6. Across 300 eligible 10 km windows, fitted slopes outside the observed cycle P10–P90 range decrease from 58 to 31. This range describes cycle variability, not statistical confidence. No case has a higher count of broad-window flags, but individual sections can worsen.

## What was held fixed
- Previously selected smoothing lambda for each case; no new smoothing search.
- Profile floor 0.05 S_ref, equal total **base** weight per cycle, existing manual bridge in case 46, and previous unit allocation with shared boundary heads.
- No new gap bridge, section slope target or manual observation exclusions.

The new objective replaces squared observation residuals by Huber loss: r² when |r| ≤ delta, otherwise 2 delta |r| − delta²; the curvature penalty is unchanged. The common automatic rule sets delta = 1.345 × 1.4826 × weighted MAD of the initial squared-fit residuals, using equal-cycle weights. Delta is calculated once per case and fixed during iteration. Effective weights become base_weight × min(1, delta/|r|); consequently, strongly inconsistent cycles may have less total effective weight after robust downweighting. There is no manual case-specific threshold selection.

IRLS alternates the observation weights and the constrained profile/offset solve. All 16 fits meet maximum parameter-update tolerance 0.1 mm; cases 15 and 54 required 207 and 175 iterations respectively. All profile gradients and final link slopes are positive. Unit boundary heads are preserved; maximum lower-bound head discrepancy is approximately 3 micrometres. Convergence does not establish observational validity.

## Case results
| Case | Low links: squared → Huber | Local windows outside cycle range: squared → Huber |
|---|---:|---:|
| 02 | 0 → 0 | 0 → 0 / 12 |
| 03 | 0 → 0 | 0 → 0 / 3 |
| 05 | 0 → 1 | 0 → 0 / 18 |
| 11 | 3 → 3 | 2 → 2 / 6 |
| 13 | 1 → 0 | 5 → 5 / 28 |
| 15 | 4 → 4 | 0 → 0 / 10 |
| 21 | 0 → 0 | 4 → 1 / 17 |
| 22 | 5 → 1 | 3 → 2 / 17 |
| 37 | 0 → 0 | 2 → 0 / 6 |
| 41 | 0 → 0 | 13 → 10 / 45 |
| 45 | 6 → 0 | 3 → 1 / 21 |
| 46 | 28 → 22 | 21 → 10 / 64 |
| 48 | 0 → 0 | 2 → 0 / 8 |
| 54 | 0 → 0 | 1 → 0 / 13 |
| 55 | 0 → 0 | 2 → 0 / 24 |
| 56 | 0 → 0 | 0 → 0 / 8 |

## Priority locations
The following are robust fitted gradients evaluated at the same observation positions as the cycle gradients; they are not always the final physical branch slope.

| Location | Cycle median | Squared loss | Huber loss |
|---|---:|---:|---:|
| 13, 63.56–66.17 km | 6.03e-5 | 6.12e-6 | 4.05e-5 |
| 22, 110–120 km | 1.24e-4 | 1.1e-5 | 1.08e-4 |
| 46, 73.65–83.14 km | 2.59e-5 | 6.50e-6 | 2.52e-5 |
| 46, 425.71–428.45 km | 4.44e-5 | 6.51e-6 | 8.62e-5 |

The first and third now fall within cycle P10–P90. Case 22 improves substantially but remains slightly below P10 (1.15e-4). The second case-46 location now exceeds P90 (7.41e-5): robust loss can overshoot locally. The new low link in case 05 is link 40, a direct-profile link; broad-window agreement there remains unchanged. Sparse-support compound domain 74 in case 46 retains the overall minimum slope. Robust loss does not supply missing observations or remove every floor-active section.

## Why interpolation alone does not resolve case 21
The observation gap extends approximately 89.21–103.38 km. No acquisition has observations spanning both sides. Each acquisition has a fitted vertical stage offset. The profile level on one disconnected side can therefore shift relative to the other while its acquisition offsets compensate, without changing its within-pass observation residuals. The overall gauge constraint fixes one common level convention, not the missing relative head difference. Smoothing and slope constraints choose among these possibilities.

Interpolation is entirely usable to connect **fixed** endpoints; it does not independently determine their relative height. For a 14.17 km gap, a 1 m versus 10 m endpoint difference gives approximately 7.1e-5 versus 7.1e-4 slope with equally straightforward interpolation. The uncertainty lies in that difference.

Here the gap head loss changes from about 10.15 m (squared-loss candidate) to 6.91 m (Huber), while whole-network drop changes from 50.75 to 47.58 m. The older production profile had about 2.35 m across this gap. These are sensitivity comparisons, not proof that the robust or production gap drop is correct. Robust fitting improves the adjoining data fit but does not create cross-gap evidence. A documented shared rule for unsupported relative levels is still needed before calling those head differences data-supported; no such new rule was imposed in this experiment.

## Interpretation and next decision
The common robust-loss rule substantially improves local slope agreement without manual case edits or a higher floor. This supports investigating it as the default fit. It does **not** yet justify replacing production slopes: smoothing was held at its squared-loss selection, and all local diagnostics use observations involved in fitting. The ordinary equal-cycle RMSE increases slightly in every case, as expected when fitting a different objective; neither RMSE alone nor the low-link count should decide the method.

Next, select smoothing under the robust objective using held-out whole cycles and compare both robust and ordinary held-out residual metrics, while retaining the local-gradient audit. Treat disconnected-support gap uncertainty separately. Then rerun unit allocation and hydraulic stability checks. No RAPID routing simulations were performed here.

## Reproduction and artifacts
- `RAPID/investigations/test_robust_slope_loss.py`: fits and old unit allocation for all 16.
- `RAPID/investigations/report_robust_slope_loss.py`: matched local cycle slopes; optional case arguments refresh selected cases.
- `RAPID/investigations/plot_robust_slope_loss.py`: figures.
- `summary.csv`, `window_summary.csv`, `windows.csv`, `priority.csv`: quantitative results.
- Per-case `floor_0.05`: profile, solver iteration history, summary and allocated link/node tables.

Plots align each acquisition by a constant offset; they show shape rather than independent absolute-level validation. Priority panels align observations locally and the squared-loss profile at the left boundary. The existing case-46 bridge is retained after fitting as in the control.


![Robust loss comparison: priority_sections.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/29274c2fe123_priority_sections.png)

![Robust loss comparison: all_profiles.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/07a481b926c9_all_profiles.png)

![Robust loss comparison: all_local_slopes.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ffe7ad3b2e13_all_local_slopes.png)

# Bounded robust-slope implementation — 2026-10-02

**Decision reference: ROBUST-SLOPES-2026-10-02.** This supersedes the earlier exploratory method recommendations in this notebook. The bounded implementation is complete for all 16 base networks; production promotion and downstream collapsed-state/re-routing work have not been performed.

## Decision
Retain the selected Huber robust fit, the 0.05 S_ref profile floor, the SWORD head-difference anchor for the disconnected gap in case 21, the documented bridge in case 46, and the existing shared-head unit allocation. Freeze this as the selected slope candidate. Hydraulic stability remains a separate work item; do not further adjust slopes simply to pass it.

## What the method does
1. Fit one shared reference profile to screened RiverSP observations, allowing a constant stage offset for each acquisition.
2. Give each cycle equal total base weight. Use Huber loss to limit the influence of extreme residuals while preserving ordinary squared-error treatment for small residuals.
3. Determine the Huber threshold from a weighted residual MAD; estimate it on training cycles only during validation. Hold it fixed while searching smoothing values.
4. Select smoothing with three held-out whole-cycle folds. Score held-out shape after removing a constant acquisition offset; retain ordinary RMSE alongside the robust score.
5. Require profile gradients of at least 0.05 times the existing static-reference global slope. This is a regularization assumption, not a measured universal minimum river slope.
6. Where observations are disconnected across a gap, constrain the missing head difference using the static SWORD reference. This applies only to case 21 in this dataset. Interpolate linearly through that gap inside the fit. Retain case 46's existing manual bridge as an internal linear constraint.
7. Transfer profile heads to the network and solve the existing owned units with fixed boundary heads and physical link lengths. Shared node heads preserve head-loss consistency. The profile floor is not an identical floor on every physical branch.

The detailed equations, parameter choices, limitations and policy for defensible manual adjustments are in `RAPID/ROBUST_REFERENCE_PROFILE_METHOD.md`. No new observation deletions or manually chosen local slopes were introduced. Future manual corrections should state the evidence, extent, constraint and resulting head-loss change.

## Numerical and observational result
- All **16 cases / 5,315 links** completed; all reference gradients and final link slopes are positive.
- **120 smoothing candidates / 360 held-cycle-fold fits**, with zero failed CV fits and no selected lambda at a search boundary. All final solvers converged.
- Links below 1e-5 decrease from **47 to 32**; none are below 1e-6. The minimum remains **3.603045e-6** in case 46's compound domain.
- Local 10 km windows outside the cycle P10–P90 slope range decrease from **58/300 to 23/300**. This is an in-sample diagnostic of local cycle agreement, not a confidence interval or independent validation.
- Earlier fixed-lambda robust loss had 31 low links and 31 window flags. The selected candidate has one more low link but fewer window flags; low-link count was not the selection objective.
- Profile constraints are satisfied to micrometre-scale head tolerances; unit boundary and shared-head checks pass. Four focused tests passed for gap identification/anchoring, incompatible references, robust outlier influence and held-out stage-offset removal.

| Case | Low links: squared → selected | Windows outside cycle range: squared → selected | Common-cap timestep feasible |
|---|---:|---:|---|
| 02 | 0 → 0 | 0 → 0 / 12 | Yes |
| 03 | 0 → 0 | 0 → 0 / 3 | Yes |
| 05 | 0 → 1 | 0 → 0 / 18 | No |
| 11 | 3 → 3 | 2 → 2 / 6 | No |
| 13 | 1 → 0 | 5 → 5 / 28 | No |
| 15 | 4 → 4 | 0 → 0 / 10 | No |
| 21 | 0 → 0 | 4 → 1 / 17 | Yes |
| 22 | 5 → 1 | 3 → 2 / 17 | No |
| 37 | 0 → 0 | 2 → 1 / 6 | Yes |
| 41 | 0 → 0 | 13 → 2 / 45 | No |
| 45 | 6 → 0 | 3 → 1 / 21 | Yes |
| 46 | 28 → 23 | 21 → 9 / 64 | No |
| 48 | 0 → 0 | 2 → 0 / 8 | Yes |
| 54 | 0 → 0 | 1 → 0 / 13 | Yes |
| 55 | 0 → 0 | 2 → 0 / 24 | Yes |
| 56 | 0 → 0 | 0 → 0 / 8 | Yes |

## Case 21: interpolation has an explicit head reference
The disconnected gap is **89.2067–103.3844 km**. The imposed SWORD head difference is **2.342809 m**, corresponding to approximately **1.65e-4 m/m** within the linear bridge. This is an explicit external-reference assumption. No acquisition measures across the gap.

The fitted total network head loss is **43.01697 m**, compared with 50.74912 m for the earlier squared-loss candidate and 44.70925 m for the older production profile. It was not selected to reproduce either previous total. Relative heights in the observed sections are fitted robustly subject to the gap constraint.

## Remaining local uncertainty
Some cycle slopes and fitted gradients still disagree. The short case-46 section around 426–428 km can remain steeper than most cycle slopes; sparse-support sections still depend on the floor. These are retained as documented limitations rather than triggering another round of hand-picked slopes. The 5% profile floor and the static gap head difference are modelling assumptions. A globally good shape fit does not establish physical validity of every link slope.

## Hydraulic preflight: a separate unresolved question
At **alpha=0.65, n=0.035, 200 m target segmentation, X=0.1**, using the stored BASED reference depth:

| Celerity treatment | Base networks with feasible integer timestep |
|---|---:|
| Uncapped | 7/16 |
| Common 0.05–2.524 m/s caps | 9/16 |
| Saved per-case caps (defaults where absent) | 10/16 |

The saved caps differ between cases; their additional pass should not be interpreted as an improvement in the profile method. Common-cap failures are **05, 11, 13, 15, 22, 41 and 46**. With saved caps the failure list excludes 22. Reference-depth recalculation at the candidate global slope is a diagnostic and gives the same primary n=.035 pass counts.

These failures mean the nominal common integer timestep interval is empty. They do not by themselves establish that the new slopes are wrong, that the full preparation safeguards fail, or that a routed hydrograph is unstable. The audit does not include adaptive safeguards, forcing-clock restrictions or all child states. It identifies the remaining hydraulic/timestep problem. Do not choose roughness, depth or slope solely to make the interval nonempty.

| Case | Controlling slow link | Slope | Width (m) | Estimated depth (m) | Required dt minimum (s) | Allowed dt maximum (s) |
|---|---:|---:|---:|---:|---:|---:|
| 05 | 91 | 2.04e-05 | 48.9 | 0.71 | 233.0 | 160.0 |
| 11 | 52 | 6.92e-06 | 107.1 | 1.37 | 285.3 | 142.6 |
| 13 | 68 | 4.36e-05 | 50.1 | 0.47 | 210.5 | 142.6 |
| 15 | 335 | 4.65e-06 | 99.4 | 0.68 | 504.7 | 142.6 |
| 22 | 882 | 0.000167 | 8.9 | 0.28 | 151.2 | 142.6 |
| 41 | 374 | 2.59e-05 | 57.9 | 1.10 | 154.8 | 150.4 |
| 46 | 67 | 4.65e-06 | 97.3 | 0.85 | 434.3 | 142.6 |

The audit also stores alpha 0.6/0.71/1.0, n 0.025/0.05/implied, and a diagnostic BASED re-evaluation. Implied n is a wide-channel Manning consistency estimate, not a fitted roughness. All scenario data are in `hydraulic_stability.csv` and `hydraulic_summary.csv`.

## Artifacts and reproduction
- `RAPID/src/rapid_tools/robust_reference_profile.py`: opt-in fitting implementation.
- `RAPID/investigations/run_bounded_robust_slopes.py`: fit, select and allocate all 16; optional case IDs for individual reruns.
- `RAPID/investigations/validate_bounded_robust_slopes.py`: saved-artifact numerical checks and input/code hashes.
- `RAPID/investigations/report_bounded_robust_slopes.py`: local cycle-slope audit.
- `RAPID/investigations/audit_bounded_robust_hydraulics.py`: base-only timestep diagnostics.
- `RAPID/investigations/plot_bounded_robust_slopes.py` and `finalize_bounded_robust_report.py`: plots and notebook report.
- Per-case `floor_0.05/`: profiles, observations/robust weights, acquisition offsets, folds, training thresholds, CV scores, explicit gap/bridge constraints and old-allocation link/node tables.
- `validation.csv`, `input_provenance.json`, `code_provenance.json`: checked results and reproducibility references.

All plots compare with the previous reselected **squared-loss / 0.05 floor** candidate, not the original production profiles. Observation curves are offset-aligned, so their vertical alignment is not independent validation of absolute level. Original notebook results are preserved as historical experiments.


![Bounded robust method: all_profiles.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ded098db41c8_all_profiles.png)

![Bounded robust method: priority_sections.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/475df6a6f4a3_priority_sections.png)

![Bounded robust method: all_local_slopes.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/3b2b309f9832_all_local_slopes.png)

![Bounded robust method: smoothing_selection.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/be1aa52e9d93_smoothing_selection.png)

# Full-distribution stability review — 2026-10-02

Follow-up to ROBUST-SLOPES-2026-10-02: inspect whether numerical conflicts arise from the slow tail, fast tail or broad hydraulic heterogeneity before deciding which assumptions to revisit. All seven failing base networks are included, with every original link in each network. No hydraulic parameters or production outputs were changed.

## Main interpretation
| Case | Distribution finding | First issue to investigate |
|---|---|---|
| 05 | Predominantly a slow-tail problem. Fast link 87 is extreme but does not exceed the raw-celerity outlier fence under either weighting. | Narrow-branch depth/width extrapolation and local low slope. Do not lower the fast side just because it minimizes changed-link count. |
| 11 | Broad, mixed distribution; conclusions depend on weighting. Four slow links are flagged with equal-link weighting, none with channel-length weighting. | Compare the long low-gradient section and shorter steep sections separately. Respect potentially real spatial regimes; avoid a network-wide percentile correction. |
| 13 | Both low and high raw-celerity tails are flagged under both weightings. | Inspect both narrow/shallow branches and steep/wide fast links. No basis for blaming only the slow side. |
| 15 | Mainly slow-tail outliers. The fast tail remains within the raw-celerity fence. | Low-depth narrow branches combined with low allocated slopes. The numerical conflict involves multiple links but a small fraction of channel length. |
| 22 | A small slow tail; the fast side is a substantial part of the distribution, with 17.4% of channel length already above the upper cap in raw celerity. | Tiny narrow branch 882 and the depth law's extrapolation there, rather than compressing the broad fast population further. |
| 41 | Predominantly slow-tail outliers; no fast raw-celerity outlier. | Narrow branch 374 and the modest relative correction required by the timestep diagnostic. |
| 46 | Both raw-celerity tails are flagged under both weightings. | Both low-slope/shallow branches and steep, deep fast reaches; use a two-sided hydraulic review rather than a single-link fix. |

These identify priorities for checking hydraulic assumptions, not evidence that flagged links are physically erroneous. A statistical extreme can be a real channel. Do not use an outlier fence as a celerity cap or depth floor without separate justification.

## Definition and weighting
The distribution screen uses log10(raw celerity): lower fence Q25 − 1.5 IQR and upper fence Q75 + 1.5 IQR, transformed back to m/s. Log scale treats multiplicative deviations symmetrically. This is a transparent descriptive screen, not a confidence interval, formal error test or universal physical threshold.

Calculate every distribution twice: equal weight per original link and weight proportional to physical channel length. The latter prevents many short links from automatically dominating. It sums all branch lengths, not reference-centerline length, and it is not discharge, conveyance, volume or hydrograph-impact weighting. Original links can generate multiple routing segments; the K value shown is that of their effective routing segment.

Use **raw celerity** to assess modeled hydraulic extremes: caps otherwise hide high values and create artificial ties. Also inspect **used K** because actual timestep compatibility depends on effective segment length and capped celerity. Neither celerity nor K alone determines whether the depth or slope estimate is plausible.

## Raw-celerity tails, using channel-length-weighted fences
| Case | Slow-tail links | Slow-tail fraction of total channel length | Fast-tail links | Fast-tail fraction of total channel length |
|---|---:|---:|---:|---:|
| 05 | 3 | 1.30% | 0 | 0.00% |
| 11 | 0 | 0.00% | 0 | 0.00% |
| 13 | 9 | 3.77% | 9 | 2.25% |
| 15 | 9 | 1.32% | 0 | 0.00% |
| 22 | 3 | 0.14% | 0 | 0.00% |
| 41 | 7 | 1.71% | 0 | 0.00% |
| 46 | 14 | 2.47% | 6 | 1.09% |

Case 11's zero flags do not imply homogeneous or well-constrained hydraulics: its wide interquartile range makes this screen permissive. Equal-link weighting flags four slow links covering 5.36% of channel length. This sensitivity is itself a reason to inspect spatial regimes rather than choose a correction from a single summary statistic.

## Size of the numerical conflicts
These are the slow/fast conflicts at the previously selected **diagnostic** integer timesteps that minimize the number of excluded intervals. These are not the same as statistical outliers, and these timesteps have not been selected for production routing.

| Case | Slow conflicts | Fraction of channel length | Fast conflicts | Fraction of channel length |
|---|---:|---:|---:|---:|
| 05 | 1 | 0.488% | 1 | 0.032% |
| 11 | 3 | 4.915% | 0 | 0.000% |
| 13 | 1 | 0.472% | 2 | 0.292% |
| 15 | 10 | 1.384% | 2 | 0.177% |
| 22 | 1 | 0.010% | 0 | 0.000% |
| 41 | 1 | 0.285% | 0 | 0.000% |
| 46 | 25 | 4.347% | 10 | 1.722% |

For example, case 15's 12 conflicting links represent approximately **1.56% of total channel length**; case 46's 35 represent **6.07%**. Counts alone overstated how broadly case 15 is affected. These fractions do not predict changes in discharge or hydrograph shape.

Case 05 illustrates why optimizing changed-link count is not sufficient: the 180 s example selects one slow and one fast link for changes, but the full distribution points primarily to unusual slow values. The fast link is not automatically the preferable target merely because it is short.

## Reading the figures
- CDF x-axis: the variable value on a logarithmic scale. Y-axis: fraction of links or channel length at or below that value.
- For K, the left tail is fast and the right tail is slow. Grey weights links equally; green weights channel length; dashed orange shows the uncapped counterpart.
- Green shading shows the compatible K interval at a diagnostic timestep: dt/1.8 ≤ K ≤ dt/0.2. It is a numerical window, not an observational uncertainty band.
- Per-case six-panel figures cover width, estimated depth, slope, raw/used celerity, used K and effective segmentation length. Red/blue ticks identify the diagnostic slow/fast conflicts; these are not statistical-outlier labels.
- Purple lines mark BASED reference width/depth. All local depths are model estimates. With common n and alpha, raw celerity follows W^(2 alpha/3) S^(1/2) within a case; the width/depth and slope panels show the ingredients of its extremes.

## Practical implication
Keep the selected slope method. Investigate whether the reference-anchored width–depth extrapolation is too strong at the slow tails of 05,15,22,41, checking slope evidence as well. Review both tails in 13 and 46. For 11, assess the distinct spatial regimes and long-reach weighting before applying any broad cap. The distributions can guide **where** to investigate; they do not select a physically correct n, alpha, depth floor or cap by themselves.

Settings: selected robust base slopes, stored BASED depth anchor, alpha=.65, n=.035, common caps .05–2.524 m/s, X=.1 and 200 m target/effective floor. All values are conditional on these assumptions. Only the seven failing networks are covered; a proposed global adjustment must subsequently be checked on all 16.

Reproduction: `RAPID/investigations/analyze_stability_distributions.py` then `report_stability_distributions.py`. Outputs include `quantiles.csv`, `tail_summary.csv`, `links_with_distribution_flags.csv`, the combined K comparison and seven six-panel distribution figures.


![Full distributions: all_K_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/d31bf7b71200_all_K_distributions.png)

![Full distributions: 05_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ff32fe95c442_05_distributions.png)

![Full distributions: 11_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/40aa3d841209_11_distributions.png)

![Full distributions: 13_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/d225324cc5a0_13_distributions.png)

![Full distributions: 15_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/d5974edb892e_15_distributions.png)

![Full distributions: 22_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ae397143b847_22_distributions.png)

![Full distributions: 41_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/032c3de69d98_41_distributions.png)

![Full distributions: 46_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/e5701d49f28c_46_distributions.png)

# Shared celerity caps and revised distributions — 2026-10-02

## Clarification and decision
The immediately preceding seven-network distribution analysis already used common **0.05–2.524 m/s** caps. The earlier hydraulic audit additionally reported saved Excel/per-case caps in a separate scenario. They were not used in that distribution analysis.

The new all-16 diagnostic uses **0.05–5.0 m/s for every base network**. The lower limit is retained because it currently clips no links; raising it to 0.1 would alter three additional links without improving the overall pass count. The upper limit is a broad, round diagnostic guardrail that affects only three links. These are explicitly listed for local review. Do not raise the shared limit simply to absorb an individual extreme, or tighten it to conceal remaining stability conflicts.

This is an operational diagnostic choice, not a universal physical maximum or a calibrated celerity model. If minimizing clipped-link count were the only objective, no clipping would trivially win: limits of 0.05–8 m/s already leave every present link unchanged. The 5 m/s threshold deliberately retains a small explicit review list. It is not claimed to be a uniquely optimal or literature-prescribed cutoff.

## Scientific interpretation of the guardrail
River wave celerity depends on the channel and event. [Allen et al. (2018)](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2018GL077914) discuss previously measured values spanning approximately 0.25–10 m/s. [HEC guidance](https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-reach-routing-methods-within-hec-hms/applying-the-muskingum-cunge-routing-method) also treats appropriate celerity as channel- and event-dependent. Therefore exceeding 5 m/s is not proof that a link's geometry or estimated hydraulics are wrong. Local corrections need evidence; the cap itself is only a transparent sensitivity/guardrail assumption.

## Common-cap comparison
All comparisons retain n=.035, alpha=.65, stored BASED reference depth, the selected robust slopes, X=.1, and the 200 m segmentation target/effective floor. All 5,315 original links from 16 base networks are included. No saved Excel cap is applied.

| Common range (m/s) | Clipped links | Clipped fraction of total channel length | Networks passing integer-timestep preflight |
|---|---:|---:|---:|
| 0.05–2.524 | 408 | 6.090% | 9/16 |
| 0.05–3 | 200 | 3.432% | 9/16 |
| 0.05–4 | 29 | 0.828% | 8/16 |
| 0.05–5 | 3 | 0.078% | 7/16 |
| 0.05–6 | 1 | 0.025% | 7/16 |
| 0.5–3 | 611 | 12.882% | 16/16 |

A grid of **265 common cap pairs** was evaluated. The least-clipping all-pass pair *in this tested grid* is 0.5–3 m/s, modifying 611 links and 12.88% of channel length; it multiplies the slowest celerity by about 6.3. This is a much larger model intervention than the selected broad guardrail. It is not recommended merely to obtain 16/16 passes. This grid result is not a proof of the globally optimal continuous cap pair.

## The only three capped links
| Case | Link | Raw celerity (m/s) | Diagnostic value | Physical link length (m) |
|---|---:|---:|---:|---:|
| 22 | 321 | 5.212 | 5.000 | 247.6 |
| 46 | 292 | 5.748 | 5.000 | 3851.0 |
| 46 | 307 | 6.176 | 5.000 | 1977.5 |

Together these are **3/5,315 links (0.056%) and 0.078% of total channel length**. No link is raised by the 0.05 lower cap. Raw values are preserved alongside diagnostic values. There have been **no edits to their slopes, depths, widths, roughness or production files**. Those input changes would require a specific justification; they have not been invented simply to meet the cap.

## Stability conflicts exposed by the broader range
Seven networks pass: **03, 21, 37, 45, 48, 55, 56**. Nine have an empty common integer-timestep interval: **02, 05, 11, 13, 15, 22, 41, 46, 54**.

| Case | Minimum intervals still requiring change | Slow + fast at an illustrative minimum-change timestep | Diagnostic timestep (s) |
|---|---:|---:|---:|
| 02 | 3 | 3 + 0 | 86 |
| 05 | 2 | 1 + 1 | 204 |
| 11 | 4 | 4 + 0 | 91 |
| 13 | 3 | 1 + 2 | 183 |
| 15 | 12 | 10 + 2 | 163 |
| 22 | 16 | 14 + 2 | 83 |
| 41 | 1 | 1 + 0 | 143 |
| 46 | 35 | 25 + 10 | 186 |
| 54 | 1 | 0 + 1 | 87 |

These are mathematical lower bounds, not an instruction to adjust every listed link. They hold the current segmentation/X and other link intervals fixed, and assume excluded links could be adjusted independently. A shared n/alpha may change many links at once and need not realize these minima. No forcing-clock divisibility or full routing run is included.

Where several timesteps have the same minimum count, the representative now minimizes the sum of physical-length-weighted absolute log celerity changes needed to enter the timestep intervals. This makes the example less arbitrary than taking the first qualifying integer, without applying those changes.

Case **54** is an integer-clock edge case: its continuous interval is **86.530–86.616 s**. Its representative 87 s diagnostic flags fast link **152** (raw celerity 4.156 m/s). Reducing that link's diagnostic celerity to 360/87 = 4.138 m/s would remove this single interval conflict, a change of about 0.44%; that adjustment has **not** been applied or justified physically. This should not drive a reduction of the common upper cap for every river.

Case **22** changes from one minimum conflict under the old cap to 16 under the broader cap, demonstrating that the old 2.524 limit was materially shaping its stability. Cases **02** and **54** newly fail because their fast links are no longer clipped at the lower upper-limit. This is greater transparency about modeled heterogeneity, not evidence that widening the cap changed river geometry.

## Revised distribution interpretation
All 16 networks now have width, estimated depth, slope, raw/used celerity, used K and effective segment-length CDFs, both equally weighted by original links and weighted by physical channel length. The raw-celerity distribution is unchanged by choosing new caps; the used K distribution and timestep conflicts change.

The previous hydraulic-tail priorities remain: slow tails in 05,15,22,41; both tails in 13,46; mixed regimes and weighting sensitivity in 11. New failure 02 also has a statistically unusual slow tail under both weighting approaches. Case 54 has statistical extremes at both ends, but its present integer failure is tiny and should be treated as such. Log-IQR flags are descriptive screens, not physical-error labels.

The green K windows correspond to diagnostic timesteps, not production clock choices. Raw values are shown so cap-created ties cannot hide fast tails. Channel-length fractions are not discharge weights.

## Artifacts and scope
- `selected_diagnostic.json`: one cap range, shared by all 16; explicitly excludes Excel overrides.
- `summary.csv`, `scenarios.csv`: the 265-pair comparison.
- `all_base_links.csv`, `stability_link_details.csv`: raw and diagnostic hydraulics.
- `cap_exception_links.csv`: the three upper-cap review candidates.
- `stability_conflict_counts.csv`, `minimum_interval_change_links.csv`: remaining interval conflicts.
- `distributions/`: all-16 quantiles, outlier flags and figures.
- Scripts: `common_celerity_cap_audit.py`, `materialize_common_cap_candidate.py`, `analyze_stability_distributions.py --run-dir RAPID/assets/bounded_robust_slopes_20261002/common_caps --cap-max 5`, and `report_common_cap_audit.py` in `RAPID/investigations`.

This reruns the diagnostic analysis, not RAPID hydrographs or child-network preparation. Production/Excel files are unchanged. Re-evaluate clipping and stability when n or alpha is finally selected; the three-link count is conditional on n=.035 and alpha=.65.


![Shared caps: cap_tradeoff.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/73a2562cffd0_cap_tradeoff.png)

![Shared caps: all_K_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/2808cfac82bf_all_K_distributions.png)

![Shared caps: 02_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/0ebac546ad09_02_distributions.png)

![Shared caps: 03_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/59c31d3c6ba8_03_distributions.png)

![Shared caps: 05_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/3941352feb9f_05_distributions.png)

![Shared caps: 11_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/52f8c37c5f45_11_distributions.png)

![Shared caps: 13_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/5bfb228129f1_13_distributions.png)

![Shared caps: 15_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/82559984932a_15_distributions.png)

![Shared caps: 21_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/806c0bbb0b42_21_distributions.png)

![Shared caps: 22_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/20fd824b5354_22_distributions.png)

![Shared caps: 37_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/b4723cd58457_37_distributions.png)

![Shared caps: 41_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/0fcf16ca0604_41_distributions.png)

![Shared caps: 45_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/30d446f4e205_45_distributions.png)

![Shared caps: 46_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ad02fbe45303_46_distributions.png)

![Shared caps: 48_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/a96d193b7987_48_distributions.png)

![Shared caps: 54_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/5fc1c1377789_54_distributions.png)

![Shared caps: 55_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/17b4d021dc14_55_distributions.png)

![Shared caps: 56_distributions.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/b9d6ba7b2a2b_56_distributions.png)

## Balanced hydraulic corrections — 2026-10-02

Reference: BALANCED-HYDRAULICS-2026-10-02. Requested after the minimum-link
correction audit, to reduce the largest required corrections by sharing them
between slow and fast links. X=0.1, segmentation, slopes, widths and the
0.05–5 m/s caps are fixed. No production changes or hydrograph runs.

## Method

For every candidate integer timestep, choose the nearest local Manning n to
0.035 that satisfies each link's capped-celerity interval. Search exhaustively
for the timestep minimizing the largest absolute log(n/0.035), then the
physical-channel-length-weighted sum of those changes. This balances
multiplicative increases and decreases; it does not minimize link count.
The depth reference is fixed and alpha is tested at 0.65, 0.625 and 0.60.

We test both exact coefficient-positivity boundaries and a 1% interior margin:
`dt >= 1.01 * 2 K X` and `dt <= 0.99 * 2 K (1-X)`.
All reported results below use the interior margin. It is a numerical buffer,
not a physical uncertainty allowance. All 192 candidate network/scenario
solutions were verified to have nonnegative Muskingum coefficients summing
to one and to satisfy their bounds. This does not establish physical validity.

## Main result

With local n constrained to 0.025–0.050, 12/16 base networks are feasible at
alpha 0.65; 13/16 at either 0.625 or 0.60. Cases 11, 15 and 46 remain
infeasible throughout these alpha tests. Case 22 becomes feasible at 0.625.
The roughness interval is an exploratory bound, not a validated range for
every link. Feasibility means some timestep and assignments exist within
that interval; unrestricted minimax assignments are tabulated below.

At alpha 0.65, allowing +/-10% local n permits 9/16 networks; +/-20% permits
11/16. At alpha 0.60 the corresponding counts are 9/16 and 11/16.
These bounded feasibility counts examine all candidate timesteps, so their
assignments need not equal the unrestricted minimax solution.

## Balanced assignments at alpha 0.65

| Case | Adjusted links | Channel length affected | Local n range |
|---|---:|---:|---:|
| 02 | 4 | 26.85% | 0.02991–0.04134 |
| 05 | 2 | 0.52% | 0.02871–0.04265 |
| 11 | 27 | 22.13% | 0.01968–0.06262 |
| 13 | 4 | 1.01% | 0.02650–0.04618 |
| 15 | 99 | 16.75% | 0.01820–0.06750 |
| 22 | 47 | 4.01% | 0.02452–0.05040 |
| 37 | 1 | 0.31% | 0.03486–0.03500 |
| 41 | 2 | 0.37% | 0.03335–0.03675 |
| 46 | 51 | 9.20% | 0.01269–0.09644 |
| 54 | 2 | 0.57% | 0.03484–0.03551 |

The other six networks need no local change with the 1% margin. Case 37
passed the earlier zero-margin audit but needs a 0.415% celerity increase on
one link to meet the added margin. Case 54 now changes two links because of
the margin; its earlier one-link 0.441% correction was exactly on a boundary.

Compared with minimizing link count, balancing avoids requiring approximately
threefold speed increases in cases 11 and 15. However, it affects many more
links: 27 and 99 respectively. Case 02 affects approximately 27% of total
physical channel length despite only four modified original links.

The controlling roughness extremes at alpha 0.65 are:
- 11: slow link 52 -> n=0.019680; fast link 14 -> n=0.062623.
- 15: slow link 335 -> n=0.018196; fast link 15 -> n=0.067500.
- 46: slow link 67 -> n=0.012686; fast link 307 -> n=0.096439.

At alpha 0.60 the corresponding network ranges remain substantial:
11: 0.02005–0.06142; 15: 0.01890–0.06496; 46: 0.01326–0.09191.

## Forcing-clock sensitivity

Production routing requires dt to divide the forcing timestep. A separate
**assumed hourly forcing** scenario restricts dt to divisors of 3600 s. It is
not a claim that every saved run uses hourly forcing. Under that restriction,
local n=0.025–0.050 gives 11/16 feasible at alpha 0.65 and 13/16 at alpha
0.625 or 0.60. Case 13 additionally misses the range at alpha 0.65.
Thus the nominal integer-only results cannot be treated as a production
preparation certificate. Actual clocks and parent-child common timestep
requirements must be checked before applying a candidate.

## Decision implication

Balanced small corrections are credible numerical candidates for the mild
cases, but are not estimates of observed roughness. Alpha 0.625 is already
enough to bring case 22 within the exploratory n interval, without going to
0.60. No tested modest alpha adjustment resolves the three difficult cases
with that n interval. Retain these as sensitivities until independent roughness
or hydraulic evidence supports the local assignments. Do not automatically
write the optimization's per-link n values into the hydraulic model.

X and segmentation have not been changed or tested as fixes in this run.
A wider local n range would be another hydraulic hypothesis to examine for
11, 15 and 46; segmentation remains a last resort. Child-state consistency
and timing/event-duration/peak-excess sensitivity remain untested here.

## Files

- `summary.csv`: all 192 network/scenario solutions, bounds and feasibility flags.
- `changed_links.csv`: every changed original link, before/after n and c,
  width, slope, depth and lengths.
- `provenance.json`: input/script hashes and fixed assumptions.
- Reproduce with `RAPID/investigations/balanced_hydraulic_corrections.py`.


[Full tables and provenance](../assets/bounded_robust_slopes_20261002/common_caps/balanced_fixes/README.md)

## Bounded local heads, then routing celerity limits — 2026-10-02

Reference: LOCAL-HEADS-CAPS-2026-10-02. Requested to keep the overall slope
method while trying small localized changes before final celerity limiting.
This is an isolated candidate investigation; no production slopes, resistance
results, parameter files or hydrograph outputs were changed.

## Final method

Keep alpha=0.65, n=0.035, X=0.1, segmentation and common 0.05–5 m/s caps.
Keep the complete fitted reference profile, all unit source/sink elevations,
all interfaces between domains and all direct-profile node elevations fixed.
Only internal endpoints of initially unstable links can move. Their incident
links share the resulting node elevation, so all path head losses telescope
consistently to unchanged boundary differences.

Test absolute node displacement budgets of 0, 2.5, 5 and 10 cm. These are
explicit numerical sensitivity limits, not measured elevation uncertainties.
No claim is made that each internal adjustment is observed by SWOT. The
conservative design does not change the longitudinal fit at locations with
steeper-cycle support: that would change unit boundary conditions.

For each network, select the integer timestep minimizing the number of
baseline corrections with a 1% interior stability margin; tie-break with the
physical-length-weighted absolute log-celerity change. Hold that clock fixed
across displacement budgets. Separately test divisors of 3600 s, explicitly
as a hypothetical hourly forcing clock.

Use linear programming to reduce normalized head-drop deficits to the
stability interval, then minimize total absolute node movement. Retain the
original slope floors (allowing only inherited solver-tolerance deviations).
No previously stable link may become unstable, and no existing correction
may become more severe. This guard prevents a solver improving one short
link by flattening an adjacent problematic link further.

Finally retain both hydraulic and routing celerity. The candidate routing
value is clipped to [1.01*2*X*L_eff/dt, 0.99*2*(1-X)*L_eff/dt], inside the
common 0.05–5 m/s envelope. This is explicit numerical regularization and
changes modeled timing/shape; it is not a newly inferred physical celerity.

## Results

| Maximum node displacement | Nominal integer: remaining limits | Hourly-divisor scenario: remaining limits |
|---|---:|---:|
| No adjustment | 82 | 90 |
| 2.5 cm | 60 | 70 |
| 5 cm | 58 | 67 |
| 10 cm | 56 | 66 |

The initial 82 differs from the earlier 77 because of the 1% interior margin
and a renewed timestep search. An interim calculation retained older
boundary-adjacent clocks and reported 100 -> 59; this final run supersedes
it, includes the no-worsening guard, and uses margin-aware clocks.

The 5 cm nominal candidate changes 53 internal nodes and 122 link slopes
(1.53% of physical channel length). It leaves 58 local routing corrections:
1.09% of original links, covering 1.46% of total physical channel length.
Minimum slope remains 3.603045e-6. Doubling the displacement limit to 10 cm
only removes two further corrections; 5 cm is the candidate for review,
not a promoted production setting.

| Case | Initial corrections | Slopes adjusted | Remaining routing limits | Largest speed increase | Largest speed reduction |
|---|---:|---:|---:|---:|---:|
| 02 | 3 | 4 | 2 | 39.0% | 0.0% |
| 05 | 2 | 0 | 2 | 16.5% | 21.6% |
| 11 | 5 | 0 | 5 | 220.1% | 0.0% |
| 13 | 3 | 3 | 3 | 17.5% | 31.8% |
| 15 | 13 | 32 | 6 | 47.9% | 16.7% |
| 22 | 17 | 39 | 9 | 58.4% | 8.3% |
| 37 | 1 | 3 | 0 | 0.0% | 0.0% |
| 41 | 1 | 3 | 0 | 0.0% | 0.0% |
| 46 | 36 | 35 | 31 | 138.4% | 61.3% |
| 54 | 1 | 3 | 0 | 0.0% | 0.0% |

Case 11 cannot be improved through the allowed internal nodes: its problematic
head drops are constrained by fixed boundaries. Case 46 remains the main
concentration of corrections. The largest remaining speed increase is
approximately 3.20x in case 11; substantial slowing also remains in case 46.
The small total affected percentage must not be presented as a small effect
on the hydrograph.

## Checks and interpretation

All 128 network/budget/clock solutions passed checks for fixed boundary heads,
node displacement limits, positive slopes, preservation of slope floors within
inherited solver tolerances, no newly unstable links, no worsened local
celerity correction, and nonnegative Muskingum coefficients summing to one
following the final routing limits.

These checks establish consistency and coefficient positivity by construction.
They are not routed-hydrograph validation. Actual saved forcing clocks,
parent-child shared clocks and all collapse states still need examination.
Any adopted slope candidate requires refreshed downstream hydraulic/resistance
calculations. Timing, event duration and peak excess sensitivity must be
reported before treating large routing corrections as acceptable.

No X or segmentation changes were made. The recommended candidate is the
bounded 5 cm internal redistribution for inspection, with all remaining
celerity exceptions explicit rather than hidden in n. Large corrections in
11 and 46 remain scientifically unresolved; this run does not endorse them.

## Reproducibility

- Script: `RAPID/investigations/bounded_local_heads_and_caps.py`.
- `summary.csv`: 128 network/budget/clock scenarios.
- `candidate_links.parquet`: full link-level values for every scenario.
- `candidate_5cm_links.csv`: nominal 5 cm candidate for convenient inspection.
- `changed_nodes.csv`: every adjusted node and its before/after elevation.
- `provenance.json`: settings and source/script hashes.


[Detailed candidate files](../assets/bounded_robust_slopes_20261002/common_caps/local_heads_and_caps/README.md)

## Explicit celerity regularization: cases 11 and 46

Reference: EXPLICIT-ROUTING-CELERITY-2026-10-02. Written after agreement to
retain hydraulic inputs and make exceptional routing corrections explicit,
then check the largest adjustments before accepting them for interpretation.

**Finding:** the safeguard provides positive-coefficient routing, but the
largest corrections are not innocuous. Whole-network timing and some selected
collapse effects remain dependent on the clock-dependent celerity limits.
Do not promote these cases as timing-robust based on the small affected count.

## Controlled comparison

- Alpha 0.65; Manning n 0.035; X 0.1; 200 m target segmentation and effective
  length floor; common celerity caps 0.05–5 m/s; 1% interior stability margin.
- Original bounded robust heads versus the saved 5 cm internal-head candidate.
  Both receive explicit routing celerity regularization.
- Case 11: base plus simple-unit 18 collapse, affecting slow link 52.
- Case 46: base plus simple-unit 24 collapse (slow link 67), and unit 105
  collapse (fast link 307). Fast connector 292 is audited in the base; there
  is no newly invented collapse for that direct connector.
- Existing child geometry and widths are retained. Node coordinates map
  exactly to the base nodes, even where child IDs were renumbered. Slopes
  are recomputed from the updated shared endpoint heads. No stale child
  slope estimates are used. Equivalent resistance is not recomputed here.
- The saved inflow samples are reconstructed as one piecewise-linear input,
  integrated to interval means. Historical zero-flow spinup is replaced by
  exact steady-state initialization. The final plateau is extended for
  settlement; the active forcing is unchanged.
- Saved forcing intervals are 690 s for case 11 and 1085 s for case 46.
  Shared routing clocks are 69/46 s and 217/155 s respectively. Each divides
  its actual saved forcing interval, and each is shared by all selected
  parent/child states and both slope treatments within its comparison.
- The first clock minimizes corrected original-link count across the tested
  original-slope states, then length-weighted log-celerity change. The second
  is the next best ranked clock differing by at least 15%. This is a
  declared two-clock sensitivity, not exhaustive convergence analysis.

## Largest corrections, original slopes

| Case/link | Hydraulic celerity | Ordinary common cap | First clock: routing celerity | Second clock: routing celerity |
|---|---:|---:|---:|---:|
| 11/52 | 0.155 m/s | 0.155 m/s | 0.646 m/s at 69 s | 0.969 m/s at 46 s |
| 46/67 | 0.092 m/s | 0.092 m/s | 0.186 m/s at 217 s | 0.261 m/s at 155 s |
| 46/307 | 6.176 m/s | 5.000 m/s | 1.642 m/s at 217 s | 2.299 m/s at 155 s |

Thus case 11/52 is accelerated by 4.18x or 6.26x. These exceed the earlier
3.2x nominal-clock result because the clock must now divide the saved forcing
interval and work for the selected child too. Case 46/307 is slowed by 67%
or 54% relative to the ordinary 5 m/s cap (more relative to hydraulic celerity).
The base-network correction counts are 9/9 for case 11 and 43/51 for case 46.

## Whole-network and collapse timing

The original-slope base peak lag is 23.94 versus 17.34 hours in case 11,
and 338.04 versus 331.01 hours in case 46. Changing the clock changes the
regularized K values as well as discretization: this is **combined clock and
regularization sensitivity**, not evidence that time discretization alone
caused these differences.

To expose binning effects, both sampled peak times and three-point quadratic
peak estimates are retained. The following child-minus-parent values use the
quadratic diagnostic, in minutes. Negative means the child peaks earlier.
Interpolation helps quantify timing but is not a claim of observational or
model accuracy at sub-timestep precision.

| Collapse | First clock, original slopes | Second clock, original slopes |
|---|---:|---:|
| 11, unit 18 | −1.28 min | −0.86 min |
| 46, unit 24 | −14.28 min | −3.84 min |
| 46, unit 105 | −2.23 min | −1.72 min |

The 5 cm slope candidate is indistinguishable for case 11. For these case 46
comparisons its quadratic peak-time effects differ by less than 0.005 minutes
from the original-slope comparison at the same clock. Apparent larger changes
in some sampled peak differences are timestep-bin effects. The slope candidate
does not remove the dependence on the regularization/clock choice.

## Event duration and peak excess

Duration uses the established plateau-to-plateau definition: first departure
from the initial plateau to sustained return to the final plateau. Thresholds
are 1% of the parent's peak-to-initial and peak-to-final amplitudes, respectively,
shared with its child. Peak excess is height above the initial plateau.

Across the selected comparisons and both slope treatments, absolute duration
changes are below 0.002%, and absolute peak-excess changes below 0.004%.
These are tiny responses for the selected collapses, not evidence that all
collapse states in cases 11 and 46 are shape-insensitive. Baseline duration
and peak height also change with clock; see the routing summaries and figures.

## Numerical checks and scope

Twenty hydrographs were computed. Frequency-domain routing uses the exact
discrete linear Muskingum recurrence on the segmented directed acyclic graph,
including width-proportional bifurcation allocation and RAPID's held external
forcing coefficient. Equal-parameter subreaches are evaluated as cascaded
transfer functions. An independently expanded split/rejoin graph agrees with
`engine.mus_rte` to machine precision. DC gain is one, flows are finite and
positive, and initial/final plateau errors are below 1e-5 of input amplitude
(in these runs near floating-point precision). This is not field validation.

No production artifacts were edited. This pilot covers three simple-unit
comparisons, not every collapse or island state. Inputs preserve existing
child geometry; downstream equivalent-resistance results still need refresh
if a slope candidate is adopted. There is no unregularized positive-coefficient
baseline at these clocks against which to establish unbiased timing.

**Recommendation:** retain the explicit safeguard as an opt-in audited method.
Keep cases 11 and 46 flagged for timing sensitivity. The present check does
not justify treating their corrected outputs as insensitive to numerical
choices or automatically rolling the method out across all production runs.

## Artifacts

- `comparisons.csv`: sampled and interpolated timing, duration and peak excess.
- `routing_summary.csv`: all 20 base/child routing checks and metrics.
- `regularized_links.csv`: each changed link with raw/common/routing celerity.
- `geometry_audit.csv`: exact-coordinate node mapping and segment counts.
- `*.parquet`: all inlet/outlet hydrographs.
- `base_hydrograph_sensitivity.png` and `paired_timing_sensitivity.png`.
- `provenance.json`: reproduction settings and impulse verification error.
- Runner: `RAPID/investigations/check_explicit_celerity_regularization.py`.
- Implementation: `RAPID/src/rapid_tools/celerity_regularization.py` and the
  opt-in `prepare_state` integration documented in
  `RAPID/EXPLICIT_ROUTING_CELERITY_METHOD.md`.


[Full report and tables](../assets/bounded_robust_slopes_20261002/common_caps/explicit_regularization_check/README.md)

![base_hydrograph_sensitivity.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/304b922cf327_base_hydrograph_sensitivity.png)

![paired_timing_sensitivity.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/f1f73690e89f_paired_timing_sensitivity.png)

## Signed-coefficient routing pilot — 2026-10-02

Reference: SIGNED-MUSKINGUM-2026-10-02. Requested after explicit timestep-based
celerity correction made results depend on the selected timestep. This test
keeps hydraulic K values fixed and separates boundedness from coefficient
positivity. It is an investigation, not a production-method promotion.

## What was held fixed

Original bounded robust slopes; alpha 0.65; n 0.035; X 0.1; 200 m segmentation
and effective length floor; common 0.05–5 m/s celerity caps. No local 5 cm head
adjustments and no additional timestep-dependent celerity corrections.
The actual forcing, child geometry, shared-head mapping and plateau event
definition are those documented in the preceding explicit-regularization pilot.

Case 11 is tested at 69, 46 and 23 seconds; case 46 at 35, 31 and 7 seconds.
Each clock divides the saved forcing interval and is shared by the selected
parent and children. All clocks satisfy dt <= 2K(1-X) for every segment, but
may violate dt >= 2KX. Fifteen actual-forcing hydrographs were computed.
At both largest and smallest clocks, EVERY base-network subreach was also
routed directly in time for actual forcing and two controlled zero-base
pulses: a 12-hour rise/24-hour fall triangle, and a 24-hour rectangular pulse.
There are 419 segments in case 11 and 6,822 in case 46. Internal child segments
were not separately audited; their outlet responses were tested at all clocks.

## Meaning of the negative allowance

The recurrence is O_new = a I_new + b I_old + c O_old, with
a=(dt-2KX)/(dt+2K(1-X)). Allowing a<0 retains the hydraulic K value but permits
an inverse initial response: a rising inflow can initially lower the outflow.
For link 11/52 at dt=69 s, a=-0.0820, b=0.1344 and c=0.9477. If local inflow
instantaneously rises from 1000 to 1100 m³/s from equilibrium, the ordinary
internal-reach recurrence gives about 991.8 m³/s initially: an artificial
8.2 m³/s dip. Actual incoming hydrographs at that link are smoother.
The external inlet segment uses RAPID's held-forcing variant of the equation.

Keeping 0<=c<1 prevents an alternating homogeneous decay and ensures that
the single-segment free response decays. It does not make the response
positivity-preserving. Shorter timesteps can make a more negative, so temporal
refinement need not remove an initial undershoot. Stability, positivity and
physical accuracy must not be used interchangeably.

This also follows from the continuous fixed-parameter Muskingum approximation:
K(1-X) dO/dt + O = I - KX dI/dt. A sufficiently abrupt positive inflow
derivative can drive the right-hand side downward. Thus convergence in dt
does not guarantee removal of this inverse-response feature of the chosen
model at fixed segmentation/K/X. Slow-varying forcing is much less demanding.

## Results: comparative timing and shape

| Case / collapse | Timesteps (s) | Peak timing change (min) | Duration change (%) | Peak-excess change (%) |
|---|---|---|---|---|
| 11 / S003_unit_18 | 69, 46, 23 | -14.55295, -14.55441, -14.55504 | -0.000572 to -0.000572 | 0.009998 to 0.009999 |
| 46 / S004_unit_24 | 35, 31, 7 | -31.66166, -31.66164, -31.66158 | -0.011271 to -0.011271 | 0.040654 to 0.040654 |
| 46 / S010_unit_105 | 35, 31, 7 | -1.53761, -1.53759, -1.53759 | -0.000021 to -0.000021 | 0.000100 to 0.000100 |

Peak timing uses the three-point quadratic estimate to avoid timestep-bin
jumps. The numerical agreement does not imply subsecond physical accuracy.
Case 11/unit 18 varies by about 0.13 seconds across the tested clocks; both
case 46 timing contrasts vary by less than 0.01 seconds. This contrasts with
the earlier clock-dependent celerity correction, especially for unit 24.
Base peak lag ranges from 27.838 to 27.882 hours in case 11 and 339.449 to
339.456 hours in case 46. Absolute lags vary more than paired contrasts.
All three tested duration/peak-excess contrasts are also effectively unchanged
at the displayed precision. This is a selected-case result, not all collapses.

## Every-segment behavior

Controlled-pulse dips are normalized by the upstream unit pulse amplitude
times the segment's steady bifurcation allocation fraction. Actual discharge
minima are absolute m³/s, across all segments. Zero-base stress inputs expose
negative perturbations which a large river baseflow could otherwise conceal.

| Case | dt (s) | Actual: minimum discharge anywhere | Smooth pulse: worst dip | Sharp pulse: worst dip |
|---|---:|---:|---:|---:|
| 11 | 23 | 18.531 m³/s | -5.30964e-06% | -0.000642328% |
| 11 | 69 | 18.531 m³/s | -2.4227e-06% | -0.000276907% |
| 46 | 7 | 2.145 m³/s | -0.00156047% | -0.537168% |
| 46 | 35 | 2.145 m³/s | -0.00151895% | -0.519237% |

No negative discharge was found in either actual-forcing base network at the
two fully audited clocks. The controlled pulses do produce small signed
transients. In case 46 the sharp pulse has a local dip of roughly half a
percent, whereas the smooth pulse has a dip of order 0.0015%. These are
localized artifacts, not growing instability; they persist under refinement.
The worst sharp-pulse segment is identified in `segment_validation.json`.
Small post-pulse oscillations/overshoots are also recorded in the segment
tables; do not characterize signed routing as strictly oscillation-free.

Integrated inflow minus trapezoidal outlet flow matches the change in the
storage represented by the implemented Muskingum recurrence. Maximum absolute
relative mass residual across the actual/pulse audits is 4.66e-12.
No negative values were clipped. Actual-input solutions remain inside the
global input range after normalization by steady branch allocation in the
coarse-clock audits; this does not prove local monotonicity at every instant.

## Important scope qualification

This is NOT permission to violate the rule on only the earlier 77 links.
At dt=35 s, 452 of 541 original links in case 46 have a negative current-inflow
coefficient; at dt=7 s, all 541 do. Many still produce entirely well-behaved
smooth-flow responses. The relevant acceptance evidence is the response and
convergence audit, not simply a small count of negative coefficients.

The evidence supports continuing with fixed-hydraulic signed-coefficient
routing as a candidate for these slow-varying events. It does not establish
positivity for abrupt/zero-flow conditions, spatial convergence, field accuracy
or performance for every real-network collapse. Production defaults and
coefficient-positivity guards remain unchanged. Before adoption, repeat the
audit for the intended full comparison set and record the event bandwidth
and observed artifact amplitudes rather than choosing an undocumented tolerance.

## Reproduction and references

- Runner: `RAPID/investigations/signed_coefficient_routing_check.py`.
- Reporter: `RAPID/investigations/report_signed_coefficient_check.py`.
- `routing_summary.csv`, `comparisons.csv`, `coefficients.csv`.
- `*_every_segment_audit.csv`: every base segment at coarse/fine clocks.
- `segment_audit_summary.csv`, `segment_validation.json`: signed dips,
  negative volume, mass residuals and independent recurrence verification.
- Outlet and worst-segment series are saved as Parquet; figures are PNG.
- The frequency-domain calculation is checked against the time-domain RAPID
  recurrence with an explicitly negative-coefficient impulse, and against
  all-segment time-domain runs on both real base networks.
- [HEC Muskingum equations](https://www.hec.usace.army.mil/confluence/hmsdocs/hmstrm/channel-flow/muskingum-model).
- [Ponce and Theurer: Accuracy Criteria in Diffusion Routing](https://ponce.sdsu.edu/accuracy_criteria_in_diffusion_routing.html),
  distinguishes numerical stability from accuracy and negative-flow artifacts.


<!-- SIGNED-MUSKINGUM-2026-10-02 -->
![paired_timing_convergence.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/7e6756979b2e_paired_timing_convergence.png)

<!-- SIGNED-MUSKINGUM-2026-10-02 -->
![negative_dip_explanation.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/ccca827ac8a4_negative_dip_explanation.png)

# Full fixed-hydraulic routing validation — 2026-10-02 to 2026-10-03

> Follow-up (2026-10-03): the provisional child reconstruction and missing-input classification below are superseded by the final **Consistent child hydraulic reconstruction** section. The five absent contexts were existing topology exclusions.

Reference: **FULL-SIGNED-ROUTING-2026-10-02**. Follow-up to SIGNED-MUSKINGUM-2026-10-02, requested to determine whether the proposed n/Kb/caps/slopes configuration is ready for production. Results remain conditional on forcing, allocation, segmentation and routing model.

## Decision

The tests support keeping the candidate hydraulic settings for further work. They do **not** establish an unconditional production-ready configuration. 76 state entries still have unresolved hydraulic inputs; cases 13/45/48 have no saved actual-event base hydrograph; production RAPID and resistance inputs still use older settings. The processable child set also contains extreme candidate head allocations (up to 60.86 m/s raw celerity), so a successful numerical run is not hydraulic acceptance. No production inputs or results were overwritten.

## Fixed settings and input refresh

n=.035, alpha=.65, D_i=D_ref(W_i/W_ref)^alpha, Kb_i=W_i/D_i, X=.1, 200 m target and effective-length floor, common .05–5 m/s caps. Keep the bounded robust reference profile and original base-unit head allocation. No local 5 cm head edits, no stability-driven n or extra celerity correction, no stability-driven effective-length increase.

For a child, exact coordinate matches inherit base heads (not renumbered node IDs). New vertices receive a constrained equal-per-link relative-slope-error fit within their incident patch. Boundary heads remain fixed; its reference slope is boundary head range divided by longest directed patch length, with the existing 1% relative floor. Where a changed topology invalidates inherited internal unit heads, the affected unit interiors can be reallocated with unit boundary/direct-profile heads protected. This is an isolated candidate refresh, not a full production hydraulic-context rebuild. Infeasible or nonpositive-head cases are excluded explicitly, never assigned stale child slopes. Case 15 uses direction130 throughout.

## Coverage and temporal convergence

| example | entries | ready | blocked | unique_states | base_segments | timesteps_s | max_timing_difference_s |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 02 | 59 | 38 | 21 | 25 | 793 | 74/10 | 0.83329 |
| 03 | 71 | 71 | 0 | 41 | 518 | 89/15 | 0.14084 |
| 05 | 96 | 96 | 0 | 52 | 1363 | 115/25 | 1.1119 |
| 11 | 44 | 44 | 0 | 27 | 419 | 69/15 | 0.17874 |
| 13 | 70 | 70 | 0 | 67 | 2436 | 100/25 | 0.12944 |
| 15 | 423 | 416 | 7 | 257 | 1990 | 85/17 | 0.30458 |
| 21 | 55 | 55 | 0 | 31 | 1292 | 66/15 | 4.2844 |
| 22 | 673 | 657 | 16 | 401 | 7566 | 55/11 | 0.18198 |
| 37 | 655 | 626 | 29 | 378 | 3238 | 5/1 | 0.0094822 |
| 41 | 293 | 293 | 0 | 199 | 5272 | 107/10 | 1.9567 |
| 45 | 95 | 95 | 0 | 94 | 1849 | 180/45 | 0.75823 |
| 46 | 396 | 395 | 1 | 224 | 6822 | 35/7 | 0.16822 |
| 48 | 75 | 75 | 0 | 73 | 695 | 60/15 | 0.70985 |
| 54 | 146 | 146 | 0 | 82 | 1191 | 70/15 | 0.47523 |
| 55 | 368 | 366 | 2 | 212 | 2770 | 154/35 | 1.5394 |
| 56 | 37 | 37 | 0 | 22 | 787 | 180/45 | 18.282 |

There are 3556 conceptual entries, 2185 unique ready hydraulic states (including 16 bases), and 2169 unique base–child comparisons tested at two clocks. Island aliases reuse identical geometry results. All registered parents are S000_base. 0 routing-screen errors were recorded. The largest two-clock paired timing difference is 18.28 s; duration and peak-excess differences reach 0.0009903 and 0.0007977 percentage points. These maxima are numerical differences, not physical accuracy claims.

Forcing is taken from each saved base event where available, cropped to one day before its active portion and initialized at exact equilibrium. End plateaus extend at least ten days or three longest-path K sums. Cases 13/45/48 use a labelled 12 h rise / 24 h fall standard triangle on unit baseflow: numerical checks only, not actual-event validation.

The exhaustive outlet screen uses the exact discrete transfer function evaluated with a band-limited representation of forcing on a 120 s reporting grid. This grid is distinct from the routing timestep. A padded smooth return outside the analyzed interval prevents circular FFT contamination; initial/final plateau errors are checked. Against native time-domain base routing, maximum waveform error fractions were 1.332e-05 at 120 s and 7.526e-06 at 60 s. Inspect `all_base_native_crosscheck.csv` for peak-time errors; small waveform errors can shift a flat maximum. Cases 21 and 56 receive additional 30 s reporting-grid / 5 s routing-clock checks (15→5 s changes the selected case-21 contrast by 0.193 s; 45→5 s changes the case-56 contrast by 1.173 s) in `selected_pair_refinement.csv`. The 5 s diagnostic need not divide saved forcing cadence because the same continuous interpolated forcing is evaluated directly.

## Every-base-segment native audit

All 39,001 base subreaches were routed with the native recurrence for their event/pulse and two zero-base stress inputs, without clipping negative values. Cases 11/46 reuse the earlier coarse native audit only after verifying identical K and segment counts; their previous fine audit remains available. Other cases use one common base-network clock below the upper bound. Internal child subreaches have **not** all received this separate native stress audit; all ready child outlets received the two-clock screen.

The minimum actual discharge across the 13 available real-event base runs is 0.80327 m³/s. The largest smooth zero-base undershoot is 0.0015189% of branch-scaled input amplitude; for an abrupt rectangular pulse it is 0.53165%. Maximum relative network mass-balance residual is 1.61e-11. Signed coefficients are therefore not positivity-preserving, even where actual-event responses remain positive. Retaining 0<=C_previous_outflow<1 gives decaying homogeneous response; it is not a physical validation or a universal nonnegative-flow guarantee.

## Child hydraulic ranges: an additional blocker

The isolated inherited-head refresh is insufficient as a production rebuild. Case 37 / S156_unit_18 reaches a raw celerity of 34.36 m/s and island_37_161 reaches 60.86 m/s; the latter assigns about 59 m of head loss to an 86 m link (slope 0.686). Case 22 / S037_unit_220 reaches 12.79 m/s. These three states exceed twice their own base-network maximum raw celerity, a triage flag rather than a proposed physical threshold. `hydraulic_review_candidates.csv` and `candidate_hydraulic_ranges.csv` retain the evidence. The minimum child slope across processable states is 2.15e-7 m/m.

The base profile is not thereby shown to be wrong: the candidate child-head transfer must be replaced/verified by a full consistent child-unit allocation before those results receive physical interpretation. Do not fix these artifacts by expanding the cap. The table label **ready** means numerically processable in this investigation only.

## The three upper-capped base links

Base links 22/321, 46/292 and 46/307 have raw celerities about 5.21, 5.75 and 6.18 m/s. Only these three base links exceed 5 m/s (3/5315 original links). `base_upper_cap_links.csv` in each case records their widths, depths, slopes and lengths. Removing the upper cap was tested for every ready unique state in cases 22 and 46 with identical clocks and other parameters; any capped child links are also released.

| example | base_peak_shift_s | max_paired_timing_shift_s | max_duration_pp | max_peak_excess_pp |
| --- | --- | --- | --- | --- |
| 22 | -1.0674 | 943.29 | 0.00088781 | 0.0010141 |
| 46 | -168.8 | 23.236 | 5.2644e-06 | 2.3716e-05 |

The largest paired effect is case 22 / CB001_bubble_9: a collapsed link is 81.8 km long with raw celerity 5.306 m/s. Removing the cap changes its timing contrast from -398.51 to -414.23 minutes (15.72 minutes, about 3.9% of the capped contrast). This is not the 12.79 m/s outlier. A small celerity correction applied over a long collapsed link can matter even when very few links are capped. Case 46 maximum paired cap effect is about 23.24 seconds.

These are **uncapped minus capped** results. They quantify the numerical effect of retaining 5 m/s; they do not independently prove the hydraulic plausibility of each raw celerity.

## Saved-method audit and remaining work

The 13 available RAPID prep manifests use n=.35, scalar BASED-reference Kb, case-dependent caps and stability effective-length correction. Cases 13/45/48 have no saved prep manifest in these roots. The saved base bubble-resistance edge tables use exactly one Kb per network in all 16 cases. Their equivalent-resistance manifests use **Chezy**, so changing Manning n does not directly change that resistance law; updating slopes and the depth/Kb geometry still matters. `saved_settings.csv` and `saved_resistance_kb_audit.csv` preserve the evidence.

Before a consistent production rerun: resolve missing contexts and failed child fits, and replace/verify the candidate allocation that creates extreme positive child slopes; rebuild and verify contexts under one frozen configuration; implement the width-dependent Kb consistently in RAPID and the chosen resistance geometry; prepare the missing actual-event runs; and make signed-coefficient routing an explicit audited policy rather than silently bypassing a preparation guard. Recompute affected equivalent-resistance results with fresh cache keys. This investigation does not justify mixing the new routed responses with old resistance tables.

The candidate settings themselves remain n=.035, alpha=.65, common .05–5 caps and the selected bounded robust slopes. The unresolved decision is how to promote and validate the numerical policy and all refreshed states, not another round of tuning n or slope to force coefficient positivity.


<!-- FULL-SIGNED-ROUTING-2026-10-02 -->
![all_case_convergence.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/5c5adf41d326_all_case_convergence.png)

<!-- FULL-SIGNED-ROUTING-2026-10-02 -->
![all_case_internal_dips.png](../assets/notebook_previews_20261002/slope_smoothing_floor_comparison_figures/cb4c330b5144_all_case_internal_dips.png)

# Consistent child hydraulic reconstruction — 2026-10-03

Reference: **CHILD-SHARED-HEADS-2026-10-03**. Written after the full signed-routing audit found failed child inputs and raw celerities up to 61 m/s. This report supersedes that audit's provisional child-head reconstruction and its classification of five island contexts as missing.

## Result

All **3,551 included entries** now reconstruct successfully across 16 examples. These represent **2,230 unique hydraulic states** and **2,214 unique base–child comparisons**. The five previously documented topology exclusions remain excluded. There are **zero reconstruction failures** among the included entries, and zero routing-screen errors. All 16 base hydraulic frames are exactly unchanged (slopes, width, depth and K).

No roughness, depth exponent, cap, segmentation, profile or external unit boundary was adjusted to obtain this result. n=.035, alpha=.65, X=.1, 200 m segmentation/effective floor and common .05–5 m/s caps remain fixed. The signed-coefficient routing policy remains an explicitly tested numerical candidate; it is not universally positivity-preserving.

## What was wrong

1. **Interior heads were being inherited too rigidly after topology changed.** A changed branch must be solved together with the interior of its owning parent unit. Keeping old internal heads or solving only newly created vertices can impose incompatible drops.
2. **The previous fallback optimization was not reliable for these scales.** In case 37 it placed interior heads close to opposite outer boundaries, concentrating roughly 59 m of head loss into an 86 m link. These extreme values were generated by the earlier investigation's reconstruction, not by the bounded robust base profile.
3. **Coordinates alone do not uniquely identify a child graph node.** Case 02 has two distinct graph nodes at the same point. Merging both into one parent ID creates a false self-loop. The new matching is injective. For coincident ports it prioritizes unchanged direct-profile continuation, then the number and physical length of unchanged incident links. Only the selected port inherits the old head; the other retains its own graph identity and receives a solved head. Unresolved ties raise an error rather than using node-ID equality or merging vertices.
4. **Five missing paths were intentional exclusions.** The source island registry explicitly says `excluded_from_primary_analysis` / `excluded_topology_review`. Those entries should never have been counted as failed hydraulic reconstruction.

## Method

Detect changed connectivity or physical link length against the base graph. Preserve unchanged domains. For an affected owned simple/compound domain, solve its surviving interior vertices and any new vertices jointly; keep its exterior/shared-profile anchors fixed. Width-only changes do not require a new slope solution. All external anchors stayed fixed in these runs; no anchor-relaxation fallback is enabled.

For each connected affected patch, minimize the existing equal-per-link relative-slope objective:

    sum_e [ (H_u - H_v) / (L_e S_ref) - 1 ]²

subject to:

    H_anchor = selected base anchor head
    (H_u - H_v) / L_e >= 0.01 S_ref

S_ref is the outer anchor head span divided by their reference-section distance span. L_e is actual link length. The existing 1% relative unit floor is retained; there is no new absolute minimum slope. The reference profile's own 5% floor is unchanged. A sparse linear solve handles feasible unconstrained optima; a convex constrained solve with primal and stationarity checks handles active constraints. The objective is scaled numerically without changing its minimizer. Hydraulic allocation contains no n, celerity cap, routing timestep or stability target.

One shared head per graph node makes path head drops telescope exactly. The independent audit checks shared heads, positive finite slopes, exact frozen boundary heads, inlet–outlet head drop, graph acyclicity and candidate CSV consistency.

## Hydraulic evidence

| example | state | previous_status | old_max_raw_celerity | new_max_raw_celerity | old_timing_minutes | new_timing_minutes |
| --- | --- | --- | --- | --- | --- | --- |
| 02 | S004_unit_5 | blocked_input | nan | 4.17024 | nan | -0.108404 |
| 02 | S023_unit_12 | blocked_input | nan | 4.17024 | nan | 3.43394 |
| 22 | S037_unit_220 | ready | 12.7854 | 5.21226 | 2.36833 | -4.65254 |
| 37 | S156_unit_18 | ready | 34.3627 | 3.11841 | 10980.1 | 3.06597 |
| 37 | island_37_161 | ready | 60.8585 | 3.13407 | 11499.3 | 38.8471 |

Across all unique states, minimum slope is **2.35589e-06 m/m**, raw celerity spans **0.0665332–6.64614 m/s**, and no link needs the 0.05 m/s lower cap. There are 11 unique state matches with a coincident-port correction. Maximum shared-head, fixed-anchor and whole-network head-drop errors are all zero at stored precision. The 16 base frames exactly reproduce the preceding base audit, so its 39,001-subreach native checks remain applicable.

The previous extreme child values should not be interpreted as physical river properties or fixed by increasing the cap. The corrected minimum being below the base minimum is possible: different branch lengths/topology allocate the same fixed boundary drop differently.

## Routing check

| example | ready_entries | excluded | unique_states | comparisons | clocks_s | min_slope | max_raw_celerity | max_timing_difference_s | max_duration_difference_pp | max_peak_difference_pp |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 02 | 59 | 0 | 36 | 35 | 74/10 | 2.22521e-05 | 4.38248 | 0.833295 | 1.76116e-06 | 0.000270616 |
| 03 | 71 | 0 | 41 | 40 | 89/15 | 7.75184e-05 | 2.14208 | 0.162049 | 8.97543e-05 | 0.000103114 |
| 05 | 96 | 0 | 52 | 51 | 115/25 | 7.35472e-06 | 2.25026 | 1.11188 | 2.50659e-05 | 2.42249e-05 |
| 11 | 44 | 0 | 27 | 26 | 69/15 | 5.53189e-06 | 4.69401 | 0.178737 | 1.38128e-06 | 8.24518e-06 |
| 13 | 70 | 0 | 67 | 66 | 100/25 | 2.70493e-05 | 3.0108 | 0.129421 | 0.000227417 | 0.000308888 |
| 15 | 422 | 1 | 263 | 262 | 119/17 | 4.64546e-06 | 2.61986 | 0.566841 | 3.01801e-05 | 8.38507e-05 |
| 21 | 55 | 0 | 31 | 30 | 66/15 | 7.12364e-05 | 4.73458 | 4.28438 | 2.17322e-05 | 1.70139e-05 |
| 22 | 671 | 2 | 409 | 408 | 55/11 | 2.35589e-06 | 5.67816 | 0.181983 | 3.79094e-05 | 2.35875e-05 |
| 37 | 654 | 1 | 397 | 396 | 5/1 | 8.05301e-06 | 3.71832 | 0.0171036 | 3.92668e-06 | 4.68184e-05 |
| 41 | 293 | 0 | 199 | 198 | 107/10 | 9.81891e-06 | 2.73207 | 1.95675 | 2.14632e-05 | 9.64115e-06 |
| 45 | 95 | 0 | 94 | 93 | 180/45 | 1.4028e-05 | 1.80826 | 0.758234 | 0.000500584 | 0.000633501 |
| 46 | 395 | 1 | 224 | 223 | 35/7 | 2.78437e-06 | 6.64614 | 0.0661329 | 1.57612e-06 | 6.98515e-06 |
| 48 | 75 | 0 | 73 | 72 | 60/15 | 0.00127937 | 4.98484 | 0.310372 | 0.000990341 | 0.000797659 |
| 54 | 146 | 0 | 82 | 81 | 70/15 | 2.89578e-05 | 4.63986 | 0.475227 | 6.23478e-05 | 0.000190766 |
| 55 | 368 | 0 | 213 | 212 | 220/55 | 5.78083e-05 | 1.52531 | 2.74283 | 2.1122e-05 | 3.32532e-05 |
| 56 | 37 | 0 | 22 | 21 | 180/45 | 4.18947e-05 | 1.79838 | 18.2823 | 0.000103614 | 2.5573e-05 |

Every unique state was screened at two common parent–child routing clocks, holding K fixed. Maximum contrast differences are **18.2823 s** for peak timing, **0.000990341 percentage points** for event duration and **0.000797659 percentage points** for peak excess. All screened outlet discharges are positive. The reporting grid is 120 s; the transfer-function method and its native-step cross-check are documented in FULL-SIGNED-ROUTING-2026-10-02. These are temporal convergence checks, not evidence of field timing accuracy. Internal child segments were not all stress-tested independently.

Thirteen cases use their saved actual-event inflow. Cases 13/45/48 have no saved actual-event base run and use the explicitly labelled standard pulse. The larger timing sensitivity in case 56 was already followed to a finer clock in the preceding audit; its remaining paired difference was approximately 1.2 s. The child reconstruction does not change that selected compound geometry.

## Cap sensitivity remains a separate issue

| example | state | uncapped_minus_capped_timing_minutes |
| --- | --- | --- |
| 22 | S000_base | 0 |
| 22 | S037_unit_220 | -1.08428e-05 |
| 22 | CB001_bubble_9 | -15.7215 |
| 46 | S000_base | 0 |
| 46 | S010_unit_105 | -0.38727 |
| 37 | S000_base | 0 |
| 37 | S156_unit_18 | 0 |
| 37 | island_37_161 | 0 |

The case-37 outliers no longer need upper capping. The long case-22 compound collapse still has a meaningful cap effect: its 81.8 km link at about 5.306 m/s gives approximately a 15.7 minute difference between capped and uncapped timing contrasts. Reconstructing heads does not remove that length-related sensitivity. Retain it in the sensitivity discussion; do not confuse it with the resolved allocation bug.

## Saved artifacts and status

- `inventory.csv`: every entry, retained exclusion and candidate manifest path; aliases share the same source context where appropriate.
- `*/contexts/*/`: link and node CSVs in the **native child node IDs**, plus manifests. Each link stores endpoint heads, slope, depth and local Kb. The manifest's `input_paths` identifies source geometry; legacy context pointers are kept separately as source provenance, never advertised as the refreshed inputs.
- `hydraulic_invariants.csv`, `base_unchanged.csv`: independent checks for every unique state and all 16 bases.
- `comparisons.csv`, `routing_summary.csv`, per-case hydrographs: recomputed paired results.
- `affected_case_changes.csv`, `targeted_cap_sensitivity.csv`: before/after and cap evidence.

Implementation: `network_variants/shared_head_reconstruction.py`; batch materialization: `RAPID/investigations/reconstruct_child_heads.py`; routing: `RAPID/investigations/route_reconstructed_child_heads.py`. Eleven focused/regression tests pass, including active-floor optimization, infeasible constraints, unchanged boundaries and coincident graph ports.

The **child reconstruction blocker is resolved for the included set**. These are saved, checked candidate contexts and rerouted investigation results. Production files, production preparation defaults and equivalent-resistance caches were not overwritten. Consistent production RAPID and equivalent-resistance preparation still needs to consume the new contexts and the agreed depth/Kb rule together; old scalar-Kb resistance results must not be paired with these new routed responses. The known three missing actual-event runs also remain a separate coverage limitation.


<!-- CHILD-SHARED-HEADS-2026-10-03 -->
![Reconstruction and routing checks](../validation/hydraulic_policy_20261003/reconstruction_summary.png)